In [ ]:
!pip install telethon pandas langdetect python-dotenv

In [ ]:
%%writefile .env
API_ID= # enter API_ID here
API_HASH= # enter API_HASH here
PHONE= # enter Phone Number here, e.g. "+60123456789"
SESSION_NAME=tele_scraper_session

In [ ]:
# Display content of .env file
!cat .env

In [ ]:
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv(".env")

API_ID = int(os.getenv("API_ID"))
API_HASH = os.getenv("API_HASH")
PHONE = os.getenv("PHONE")
SESSION_NAME = os.getenv("SESSION_NAME", "tele_scraper_session")

print(API_ID, API_HASH[:5] + "...", PHONE, SESSION_NAME)

In [ ]:
from telethon import TelegramClient
import os
from dotenv import load_dotenv

load_dotenv()

api_id = os.getenv("API_ID")
api_hash = os.getenv("API_HASH")
phone = os.getenv("PHONE")

In [ ]:
client = TelegramClient("session_name", api_id, api_hash)

await client.start(phone=phone)


In [ ]:
# List all groups that we've joined to get the group ID
async def list_groups():
    async for dialog in client.iter_dialogs():
        if dialog.is_group:  # filters only groups
            print(f"Group Name: {dialog.name} | ID: {dialog.id}")

await list_groups()

In [ ]:
!pip install telethon python-dotenv pandas nest_asyncio

import os
import pandas as pd
import nest_asyncio
from dotenv import load_dotenv
from telethon import TelegramClient

# Allow nested event loops
nest_asyncio.apply()

# 1. Load credentials
load_dotenv()

api_id = int(os.getenv("API_ID"))
api_hash = os.getenv("API_HASH")
phone = os.getenv("PHONE")

# 2. Start client
client = TelegramClient("session_name", api_id, api_hash)

async def main():
    await client.start(phone=phone)

    # 3. Choose group ID
    group_id =  # replace with the group ID that we want to scrape

    # 4. Fetch all text messages in batches
    messages = []
    batch_size = 200
    offset_id = 0

    while True:
        batch = []
        async for msg in client.iter_messages(group_id, limit=batch_size, offset_id=offset_id):
            if msg.text and msg.text.strip():  # only keep text messages
                batch.append({
                    "id": msg.id,
                    "date": msg.date,
                    "sender": msg.sender_id,
                    "text": msg.text.strip()
                })

        if not batch:
            break

        messages.extend(batch)
        offset_id = batch[-1]["id"]
        print(f"Fetched {len(messages)} text messages so far...")

    # 5. Save to CSV
    df = pd.DataFrame(messages)
    df.to_csv("telegram_messages.csv", index=False, encoding="utf-8")
    print(f"All text messages saved to telegram_messages.csv ({len(messages)} total)")

# Run safely inside Colab
await main()


In [ ]:
await client.disconnect()